UP LOAD FILE TO HTPPS

In [ ]:
import requests
import os

# --- ตั้งค่าการเชื่อมต่อ ---
SERVER_IP = "100.85.77.73"
PORT = "8001"
# ต้องระบุ Path ให้ตรงกับที่เขียนไว้ใน FastAPI
URL = f"http://{SERVER_IP}:{PORT}/api/upload" 

# --- ตั้งค่าไฟล์และโฟลเดอร์ ---
FILE_NAME = r"D:\Project_end\New_world\my_project\data\raw\pwm_duty_0.20_freq_0.10_pwm_amplitude_1_freq_0.1_duty_0.2.csv"
TARGET_FOLDER = "laptop-02/logs"

def start_upload():
    if not os.path.exists(FILE_NAME):
        print(f"❌ ไม่พบไฟล์ตามที่ระบุ: {FILE_NAME}")
        return

    print(f"📤 กำลังส่งไฟล์ไปที่: {URL}")
    print(f"📂 ปลายทางบน Server: {TARGET_FOLDER}")

    try:
        with open(FILE_NAME, 'rb') as f:
            files = {'file': f}
            data = {'folder': TARGET_FOLDER}
            
            # ส่งคำขอ POST ไปยัง /api/upload
            response = requests.post(URL, files=files, data=data)
            
            if response.status_code == 200:
                print("✅ อัปโหลดสำเร็จ!")
                print("Server Response:", response.json())
            else:
                print(f"⚠️ อัปโหลดไม่สำเร็จ (Status Code: {response.status_code})")
                print("Error:", response.text)
                
    except Exception as e:
        print(f"❌ เชื่อมต่อกับ Server ไม่ได้: {e}")

if __name__ == "__main__":
    start_upload()

WATCH UPDATE FILE

In [ ]:
import requests
import os
import time

# --- ตั้งค่าการเชื่อมต่อ ---
SERVER_IP = "100.85.77.73"
PORT = "8001"
BASE_URL = f"http://{SERVER_IP}:{PORT}"
API_LIST = f"{BASE_URL}/api/list-files"
STORAGE_URL = f"{BASE_URL}/storage" # URL สำหรับโหลดไฟล์

# --- ตั้งค่าโฟลเดอร์ใน Laptop ---
LOCAL_SAVE_PATH = r"D:\Server_Sync_Folder" 
TARGET_SERVER_FOLDER = "laptop-02/logs" # โฟลเดอร์บน server ที่ต้องการเฝ้าดู

if not os.path.exists(LOCAL_SAVE_PATH):
    os.makedirs(LOCAL_SAVE_PATH)

def sync_files():
    print(f"🔍 Checking for updates at {time.strftime('%H:%M:%S')}...")
    try:
        # 1. ขอรายชื่อไฟล์จาก Server
        response = requests.get(API_LIST, params={"folder": TARGET_SERVER_FOLDER})
        if response.status_code != 200:
            print("❌ ไม่สามารถดึงข้อมูลจาก Server ได้")
            return

        server_items = response.json().get("items", [])

        for item in server_items:
            if item["is_dir"]: continue # ข้ามถ้าเป็นโฟลเดอร์

            filename = item["name"]
            local_file_path = os.path.join(LOCAL_SAVE_PATH, filename)

            # 2. ตรวจสอบว่าไฟล์นี้มีในเครื่องหรือยัง
            # (หรือเช็คขนาดไฟล์/เวลาแก้ไขเพื่อความแม่นยำขึ้นได้)
            if not os.path.exists(local_file_path):
                print(f"✨ พบไฟล์ใหม่: {filename} -> กำลังดาวน์โหลด...")
                
                # 3. สั่ง Download
                download_url = f"{STORAGE_URL}/{TARGET_SERVER_FOLDER}/{filename}"
                file_data = requests.get(download_url)
                
                if file_data.status_code == 200:
                    with open(local_file_path, "wb") as f:
                        f.write(file_data.content)
                    print(f"✅ ดาวน์โหลดสำเร็จ: {filename}")
                else:
                    print(f"⚠️ โหลดไฟล์ {filename} ไม่สำเร็จ")

    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาด: {e}")

if __name__ == "__main__":
    print("🚀 เริ่มระบบเฝ้าดูไฟล์ (Sync System)...")
    while True:
        sync_files()
        time.sleep(10) # พัก 10 วินาทีแล้วเช็คใหม่ (ปรับเปลี่ยนเวลาได้)

hybrid pooling

In [4]:
import requests
import os
import time
from flask import Flask, request
from threading import Thread

app = Flask(__name__)

# --- Configuration ---
SERVER_IP = "100.85.77.73"
PORT = "8001"
BASE_URL = f"http://{SERVER_IP}:{PORT}"
LOCAL_SAVE_PATH = r"D:\Server_Sync_Folder"
TARGET_SERVER_FOLDER = "laptop-02/logs"

if not os.path.exists(LOCAL_SAVE_PATH):
    os.makedirs(LOCAL_SAVE_PATH)

def download_logic(filename, folder):
    """ฟังก์ชันหลักในการดาวน์โหลดไฟล์"""
    local_path = os.path.join(LOCAL_SAVE_PATH, filename)
    
    # เช็คว่าไฟล์มีอยู่แล้วและขนาดเท่ากันไหม (ป้องกันการโหลดซ้ำโดยไม่จำเป็น)
    download_url = f"{BASE_URL}/storage/{folder}/{filename}"
    
    try:
        # ดึง Metadata ของไฟล์มาเช็คขนาดก่อนโหลด
        head = requests.head(download_url)
        server_size = int(head.headers.get('content-length', 0))
        
        if os.path.exists(local_path):
            if os.path.getsize(local_path) == server_size:
                return # ไฟล์เหมือนกันเป๊ะ ข้ามไป
        
        print(f"📥 Downloading: {filename} ({server_size} bytes)")
        r = requests.get(download_url)
        with open(local_path, "wb") as f:
            f.write(r.content)
        print(f"✅ Saved: {filename}")
    except Exception as e:
        print(f"❌ Download Error: {e}")

def run_full_polling():
    """ระบบ Polling: ตรวจสอบไฟล์ทั้งหมดบน Server"""
    print(f"🔍 [Polling] Checking all files at {time.strftime('%H:%M:%S')}")
    try:
        response = requests.get(f"{BASE_URL}/api/list-files", params={"folder": TARGET_SERVER_FOLDER})
        if response.status_code == 200:
            items = response.json().get("items", [])
            for item in items:
                if not item["is_dir"]:
                    download_logic(item["name"], TARGET_SERVER_FOLDER)
    except Exception as e:
        print(f"⚠️ Polling failed: {e}")

# --- Interrupt Listener (Webhook) ---
@app.route('/trigger-sync', methods=['POST'])
def trigger_sync():
    data = request.json
    filename = data.get('filename')
    folder = data.get('folder')
    print(f"🔔 [Interrupt] New file notification: {filename}")
    # รันการโหลดทันทีเมื่อถูกขัดจังหวะ
    Thread(target=download_logic, args=(filename, folder)).start()
    return {"status": "received"}, 200

def polling_loop():
    """ลูปสำหรับรัน Polling ทุกๆ 5 นาที"""
    while True:
        run_full_polling()
        time.sleep(300) # 5 นาที

if __name__ == '__main__':
    # 1. รันระบบ Polling เป็น Background Thread
    Thread(target=polling_loop, daemon=True).start()
    
    # 2. รัน Receiver รอรับ Interrupt (พอร์ต 9000)
    print("🚀 Hybrid Sync System Started!")
    print(f"📍 Local Storage: {LOCAL_SAVE_PATH}")
    app.run(host='0.0.0.0', port=9000)

🔍 [Polling] Checking all files at 17:02:53🚀 Hybrid Sync System Started!
📍 Local Storage: D:\Server_Sync_Folder

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:9000
 * Running on http://10.20.10.96:9000
Press CTRL+C to quit


⚠️ Polling failed: HTTPConnectionPool(host='100.85.77.73', port=8001): Max retries exceeded with url: /api/list-files?folder=laptop-02%2Flogs (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001B25ABCA930>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))


🔍 [Polling] Checking all files at 17:07:55
🔍 [Polling] Checking all files at 17:12:55
🔍 [Polling] Checking all files at 17:17:55
🔍 [Polling] Checking all files at 17:22:56
🔍 [Polling] Checking all files at 17:27:56
🔍 [Polling] Checking all files at 17:32:56
🔍 [Polling] Checking all files at 17:37:56
🔍 [Polling] Checking all files at 17:42:56
🔍 [Polling] Checking all files at 17:47:56
🔍 [Polling] Checking all files at 17:52:56
🔍 [Polling] Checking all files at 17:57:56
🔍 [Polling] Checking all files at 18:02:56
🔍 [Polling] Checking all files at 18:07:57
🔍 [Polling] Checking all files at 18:12:57
🔍 [Polling] Checking all files at 18:17:57
🔍 [Polling] Checking all files at 18:22:57
🔍 [Polling] Checking all files at 18:27:57
🔍 [Polling] Checking all files at 18:32:57
🔍 [Polling] Checking all files at 18:37:58
🔍 [Polling] Checking all files at 18:42:58
🔍 [Polling] Checking all files at 18:47:58
🔍 [Polling] Checking all files at 18:52:58
🔍 [Polling] Checking all files at 18:57:58
🔍 [Polling]

hybrid sync

In [5]:
import requests
import os
import time
from flask import Flask, request
from threading import Thread

app = Flask(__name__)

# --- ⚙️ การตั้งค่า (ปรับแต่งที่นี่) ---
SERVER_IP = "100.85.77.73"  # <--- เปลี่ยนเป็น IP ของ Ubuntu Server
SERVER_PORT = "8001"
BASE_URL = f"http://{SERVER_IP}:{SERVER_PORT}"
LOCAL_SAVE_PATH = r"D:\Server_Sync_Folder" # <--- โฟลเดอร์ที่อยากให้ไฟล์ไปลง
TARGET_SERVER_FOLDER = "laptop-02/logs"    # <--- โฟลเดอร์บน Server ที่จะเฝ้าดู

# สร้างโฟลเดอร์ในเครื่องถ้ายังไม่มี
if not os.path.exists(LOCAL_SAVE_PATH):
    os.makedirs(LOCAL_SAVE_PATH)

# --- 🛠 ฟังก์ชันหลัก ---

def download_file(filename, folder):
    """ฟังก์ชันดาวน์โหลดไฟล์จาก Server ลง Laptop"""
    local_path = os.path.join(LOCAL_SAVE_PATH, filename)
    download_url = f"{BASE_URL}/storage/{folder}/{filename}"
    
    try:
        # 1. ตรวจสอบ Metadata ก่อน (ไม่ต้องโหลดถ้าไฟล์เหมือนเดิม)
        head = requests.head(download_url, timeout=5)
        server_size = int(head.headers.get('content-length', 0))
        
        if os.path.exists(local_path):
            if os.path.getsize(local_path) == server_size:
                # print(f"⏩ Skip: {filename} (Already exists)")
                return

        # 2. เริ่มดาวน์โหลด
        print(f"📥 Downloading: {filename}...")
        r = requests.get(download_url, stream=True)
        if r.status_code == 200:
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✅ Successfully Saved: {filename}")
        else:
            print(f"⚠️ Failed to download: {filename} (Status: {r.status_code})")
            
    except Exception as e:
        print(f"❌ Error during download {filename}: {e}")

def run_polling():
    """ระบบสำรอง: ตรวจสอบไฟล์ทั้งหมดทุกๆ 5 นาที (เผื่อเน็ตหลุดช่วง Interrupt)"""
    while True:
        print(f"🔍 [Polling] Checking for updates at {time.strftime('%H:%M:%S')}...")
        try:
            response = requests.get(f"{BASE_URL}/api/list-files", params={"folder": TARGET_SERVER_FOLDER}, timeout=10)
            if response.status_code == 200:
                items = response.json().get("items", [])
                for item in items:
                    if not item["is_dir"]:
                        download_file(item["name"], TARGET_SERVER_FOLDER)
        except Exception as e:
            print(f"📡 Polling: Server unreachable. Retrying in 5 mins...")
        
        time.sleep(300) # รอ 5 นาที

# --- 🔔 ระบบ Interrupt (Webhook Receiver) ---

@app.route('/trigger-sync', methods=['POST'])
def trigger_sync():
    """รอรับสัญญาณ 'ดีดนิ้ว' จาก Server เมื่อมีการอัปโหลดใหม่"""
    data = request.json
    filename = data.get('filename')
    folder = data.get('folder')
    
    # ถ้าโฟลเดอร์ที่อัปโหลดตรงกับที่เราสนใจ หรือถ้าเราอยากโหลดทุกโฟลเดอร์
    if folder == TARGET_SERVER_FOLDER or TARGET_SERVER_FOLDER == "":
        print(f"🔔 [Interrupt] New file detected: {filename}")
        # สั่งโหลดทันทีใน Thread ใหม่เพื่อไม่ให้ Server ต้องรอ
        Thread(target=download_file, args=(filename, folder)).start()
        
    return {"status": "accepted"}, 200

if __name__ == '__main__':
    # 1. เริ่มระบบ Polling (Background)
    Thread(target=run_polling, daemon=True).start()
    
    # 2. เริ่มระบบ Interrupt Receiver (Port 9000)
    print("🚀 Laptop Sync Client is Running!")
    print(f"📍 Watching: {TARGET_SERVER_FOLDER} on Server")
    print(f"📍 Saving to: {LOCAL_SAVE_PATH}")
    app.run(host='0.0.0.0', port=9000)

🔍 [Polling] Checking for updates at 20:56:01...🚀 Laptop Sync Client is Running!
📍 Watching: laptop-02/logs on Server
📍 Saving to: D:\Server_Sync_Folder

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:9000
 * Running on http://10.20.10.96:9000
Press CTRL+C to quit
100.85.77.73 - - [26/Dec/2025 20:56:18] "POST /trigger-sync HTTP/1.1" 200 -


🔔 [Interrupt] New file detected: pwm_duty_0.00_freq_0.10_pwm_amplitude_1_freq_0.1_duty_0.0.png
📥 Downloading: pwm_duty_0.00_freq_0.10_pwm_amplitude_1_freq_0.1_duty_0.0.png...
✅ Successfully Saved: pwm_duty_0.00_freq_0.10_pwm_amplitude_1_freq_0.1_duty_0.0.png


🔍 [Polling] Checking for updates at 21:01:02...
🔍 [Polling] Checking for updates at 21:06:02...
🔍 [Polling] Checking for updates at 21:11:02...
🔍 [Polling] Checking for updates at 21:16:02...
🔍 [Polling] Checking for updates at 21:21:02...
🔍 [Polling] Checking for updates at 21:26:03...
🔍 [Polling] Checking for updates at 21:31:03...
🔍 [Polling] Checking for updates at 21:36:03...
🔍 [Polling] Checking for updates at 21:41:03...
🔍 [Polling] Checking for updates at 21:46:03...
🔍 [Polling] Checking for updates at 21:51:03...
🔍 [Polling] Checking for updates at 21:56:04...
🔍 [Polling] Checking for updates at 22:01:04...
🔍 [Polling] Checking for updates at 22:06:04...
🔍 [Polling] Checking for updates at 22:11:04...
🔍 [Polling] Checking for updates at 22:16:04...
🔍 [Polling] Checking for updates at 22:21:04...
🔍 [Polling] Checking for updates at 22:26:04...
🔍 [Polling] Checking for updates at 22:31:04...
🔍 [Polling] Checking for updates at 22:36:05...
🔍 [Polling] Checking for updates at 22:4

download , delete , sync

In [ ]:
import requests
import os
import time
from flask import Flask, request
from threading import Thread

app = Flask(__name__)

# --- ⚙️ การตั้งค่า ---
SERVER_IP = "100.85.77.73"  # <--- เปลี่ยนเป็น IP ของ Ubuntu Server
SERVER_PORT = "8001"     # <--- พอร์ตของ FastAPI
BASE_URL = f"http://{SERVER_IP}:{SERVER_PORT}"
LOCAL_SAVE_PATH = r"D:\Server_Sync_Folder" 
TARGET_SERVER_FOLDER = "laptop-02/logs"

if not os.path.exists(LOCAL_SAVE_PATH): os.makedirs(LOCAL_SAVE_PATH)

# --- 🛠 ส่วนของฟังก์ชันการทำงาน (Core Logic) ---

def download_file(filename, folder):
    """ดาวน์โหลดไฟล์ลงเครื่อง"""
    local_path = os.path.join(LOCAL_SAVE_PATH, filename)
    url = f"{BASE_URL}/storage/{folder}/{filename}"
    try:
        r = requests.get(url, stream=True, timeout=10)
        if r.status_code == 200:
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
            print(f"\n✅ Downloaded: {filename}")
        else: print(f"\n❌ Server Error (Download): {r.status_code}")
    except Exception as e: print(f"\n❌ Error: {e}")

def upload_to_server(file_path):
    """อัปโหลดไฟล์จาก Laptop ขึ้น Server"""
    if not os.path.exists(file_path):
        print("❌ File not found locally.")
        return
    
    filename = os.path.basename(file_path)
    try:
        with open(file_path, 'rb') as f:
            files = {'file': (filename, f)}
            data = {'folder': TARGET_SERVER_FOLDER}
            r = requests.post(f"{BASE_URL}/api/upload", files=files, data=data)
            print(f"📤 Upload Response: {r.json()}")
    except Exception as e: print(f"❌ Upload Error: {e}")

def delete_on_server(filename):
    """สั่งลบไฟล์บน Server"""
    try:
        payload = {"filename": filename, "folder": TARGET_SERVER_FOLDER}
        r = requests.post(f"{BASE_URL}/api/delete", json=payload)
        print(f"🗑️ Delete Response: {r.json()}")
    except Exception as e: print(f"❌ Delete Error: {e}")

# --- 🔔 ระบบ Webhook Receiver (รอรับ Action จาก Server) ---

@app.route('/trigger-sync', methods=['POST'])
def trigger_event():
    data = request.json
    action = data.get('action') # 'sync' หรือ 'delete'
    filename = data.get('filename')
    folder = data.get('folder')

    if folder == TARGET_SERVER_FOLDER or TARGET_SERVER_FOLDER == "":
        if action == "sync":
            print(f"\n🔔 [Signal] Server added: {filename}. Downloading...")
            Thread(target=download_file, args=(filename, folder)).start()
        elif action == "delete":
            print(f"\n🔔 [Signal] Server deleted: {filename}. Removing local copy...")
            local_path = os.path.join(LOCAL_SAVE_PATH, filename)
            if os.path.exists(local_path): os.remove(local_path)
    return {"status": "ok"}, 200

# --- 🎮 Interactive Menu (สำหรับการทดสอบ) ---

def manual_menu():
    time.sleep(2) # รอให้ Flask รันขึ้นมาก่อน
    while True:
        print("\n" + "="*30)
        print(f"💻 LAPTOP SYNC MENU (Folder: {TARGET_SERVER_FOLDER})")
        print("="*30)
        print("1. 📤 Upload file to Server")
        print("2. 🗑️ Delete file from Server")
        print("3. 📁 List local files")
        print("0. 🚪 Exit")
        choice = input("Enter choice: ")

        if choice == '1':
            path = input("Enter full path of file to upload: ").strip('"')
            upload_to_server(path)
        elif choice == '2':
            name = input("Enter filename to delete from Server: ")
            delete_on_server(name)
        elif choice == '3':
            print(f"Local files in {LOCAL_SAVE_PATH}:", os.listdir(LOCAL_SAVE_PATH))
        elif choice == '0':
            os._exit(0)

if __name__ == '__main__':
    # รัน Menu ใน Thread แยก
    Thread(target=manual_menu, daemon=True).start()
    
    # รัน Flask รับสัญญาณ
    print("🚀 Sync Client Listening on port 9000...")
    app.run(host='0.0.0.0', port=9000, debug=False, use_reloader=False)